# Лабораторная работа 5
## Двойственный симплекс-метод

Реализация двойственного симплекс-метода для задачи линейного программирования в канонической форме.


In [ ]:
import numpy as np


def dual_simplex(c, A, b, B_init, max_iter=1000, eps=1e-9):
    c = np.array(c, dtype=float)
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    B = list(B_init)

    m, n = A.shape
    history = []

    for iteration in range(1, max_iter + 1):
        A_B = A[:, B]
        try:
            A_B_inv = np.linalg.inv(A_B)
        except np.linalg.LinAlgError as exc:
            raise ValueError("Матрица A_B вырождена. Базис некорректен.") from exc

        c_B = c[B]
        y = c_B @ A_B_inv

        x_B = A_B_inv @ b
        x = np.zeros(n)
        for i, idx in enumerate(B):
            x[idx] = x_B[i]

        history.append({
            "iter": iteration,
            "B": B.copy(),
            "x": x.copy(),
            "y": y.copy(),
        })

        # Шаг 5: если псевдоплан неотрицателен, найден оптимум
        if np.all(x_B >= -eps):
            x[np.abs(x) < eps] = 0.0
            return "optimal", x, B, history

        # Шаг 6: выбираем отрицательную компоненту с минимальным индексом j_k
        negative_candidates = [
            (B[pos], pos) for pos in range(m) if x_B[pos] < -eps
        ]
        j_k, k = min(negative_candidates, key=lambda pair: pair[0])

        # Шаг 7
        delta_y = A_B_inv[k, :]  # k-я строка A_B^{-1}
        N = [j for j in range(n) if j not in B]
        mu = {j: float(delta_y @ A[:, j]) for j in N}

        # Шаг 8
        if all(mu_j >= -eps for mu_j in mu.values()):
            return "infeasible", None, B, history

        # Шаги 9-10: минимум sigma_j, при равенстве - минимальный индекс
        sigma_candidates = []
        for j in N:
            if mu[j] < -eps:
                reduced_cost = c[j] - A[:, j] @ y
                sigma = reduced_cost / mu[j]
                sigma_candidates.append((sigma, j))

        sigma0, j0 = min(sigma_candidates, key=lambda item: (item[0], item[1]))

        history[-1]["k"] = k
        history[-1]["j_out"] = j_k
        history[-1]["mu"] = mu
        history[-1]["sigma0"] = sigma0
        history[-1]["j_in"] = j0

        # Шаг 11
        B[k] = j0

    raise RuntimeError("Превышено максимальное число итераций.")


In [ ]:
# x1 + x2 -> max
# 2x1 + x2 - x3 = 3
# -2x1 - x2 - x4 = -6
# -x1 - 2x2 - x5 = -6
# x_i >= 0

c = np.array([1.0, 1.0, 0.0, 0.0, 0.0])
A = np.array([
    [ 2.0,  1.0, -1.0,  0.0,  0.0],
    [-2.0, -1.0,  0.0, -1.0,  0.0],
    [-1.0, -2.0,  0.0,  0.0, -1.0],
])
b = np.array([3.0, -6.0, -6.0])

# Начальный базис: B = {1, 3, 4} (1-based)
B_init = [0, 2, 3]  # 0-based

status, x_opt, B_opt, history = dual_simplex(c, A, b, B_init)

print("Статус:", status)
print("\nИстория итераций:")
for record in history:
    B_1based = [idx + 1 for idx in record["B"]]
    print(f"Итерация {record['iter']}: B = {B_1based}, x^T = {np.round(record['x'], 6)}")

if status == "optimal":
    print("\nОптимальный план:")
    print("x^T =", np.round(x_opt, 6))
    print("B (0-based) =", B_opt)
    print("B (1-based) =", [idx + 1 for idx in B_opt])
    print("Значение целевой функции:", float(c @ x_opt))
else:
    print("Задача несовместна для данного базиса в рамках двойственного симплекс-метода.")


Статус: optimal

История итераций:
Итерация 1: B = [1, 3, 4], x^T = [ 6.  0.  9. -6.  0.]
Итерация 2: B = [1, 3, 2], x^T = [2. 2. 3. 0. 0.]

Оптимальный план:
x^T = [2. 2. 3. 0. 0.]
B (0-based) = [0, 2, 1]
B (1-based) = [1, 3, 2]
Значение целевой функции: 4.0


In [ ]:
# -4x1 - 3x2 - 7x3 -> max
# -2x1 - x2 - 4x3 + x4      = -1
# -2x1 - 2x2 - 2x3      + x5 = -3/2
# x_i >= 0

c3 = np.array([-4.0, -3.0, -7.0, 0.0, 0.0])
A3 = np.array([
    [-2.0, -1.0, -4.0, 1.0, 0.0],
    [-2.0, -2.0, -2.0, 0.0, 1.0],
])
b3 = np.array([-1.0, -1.5])

# Начальный базис: B = (4, 5) (1-based)
B3_init = [3, 4]  # 0-based

status3, x3_opt, B3_opt, history3 = dual_simplex(c3, A3, b3, B3_init)

print("Статус:", status3)
for record in history3:
    B_1based = [idx + 1 for idx in record["B"]]
    print(f"Итерация {record['iter']}: B = {B_1based}, x^T = {np.round(record['x'], 6)}")

if status3 == "optimal":
    print("\nОптимальный план:", np.round(x3_opt, 6))
    print("B (1-based):", [idx + 1 for idx in B3_opt])
    print("Значение целевой функции:", float(c3 @ x3_opt))


Статус: optimal
Итерация 1: B = [4, 5], x^T = [ 0.   0.   0.  -1.  -1.5]
Итерация 2: B = [3, 5], x^T = [ 0.    0.    0.25  0.   -1.  ]
Итерация 3: B = [3, 1], x^T = [ 1.    0.   -0.25  0.    0.  ]
Итерация 4: B = [2, 1], x^T = [0.25 0.5  0.   0.   0.  ]

Оптимальный план: [0.25 0.5  0.   0.   0.  ]
B (1-based): [2, 1]
Значение целевой функции: -2.5
